In [27]:
import torch
import torch.nn as nn 
import torch.optim as optim 
import torchvision 
import torchvision.transforms as transforms 
from torch.utils.data import DataLoader, Subset

In [28]:
class CNNDropout(nn.Module): 
    def __init__(self): 
        super(CNNDropout, self).__init__()

    #first block for eyeing the image 
        self.convb1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), 
            nn.BatchNorm2d(32),
            nn.ReLU(), 
            nn.MaxPool2d(2)
        )
        
        #second block looks deeper same as the prev one but looks for 32 more feats 
        self.convb2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1), 
            nn.BatchNorm2d(64), 
            nn.ReLU(), 
            nn.MaxPool2d(2)
        )
        
        self.flatten = nn.Flatten()
        #4096 layers condensed to 512 layers
        self.dense1 = nn.Linear(64*8*8, 512)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        #final layer condenses the 512 to 10 as to what is to be predicted in CIFAR 10 Dataset
        self.dense2 = nn.Linear(512, 10)
    
    def forward(self, x): 
        x = self.convb1(x)
        x = self.convb2(x)
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.dense2(x)
        return x 
    

In [29]:
#img converted to pytorch sensors
transform = transforms.Compose([transforms.ToTensor()])
train_d = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_d = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Made subsets of the data for faster training 

In [30]:
train_s = Subset(train_d, range(200))
test_s = Subset(test_d, range(50))

### Use Dataloader as the model is not training on the whole dataset, but only on a subset of it. This is done to speed up the training process and to avoid overfitting. The Dataloader will load the data in batches, which will be used for training the model.

In [31]:
train_l = DataLoader(train_s, batch_size=10, shuffle= True)
test_l = DataLoader(test_s, batch_size=10, shuffle= False)

In [32]:
def train(model, opt, crit,  n_e, device): 
    model.to(device)
    
    for e in range(n_e): 
        model.train()
        tr_l = 0
        c_t = 0 
        t_t = 0 
        
        for d, t in train_l: 
            d, t = d.to(device), t.to(device)
            #remove all predictions 
            opt.zero_grad()
            #make new predictions
            op = model(d)
            loss = crit(op,t)
            #go backward in the nueral net to see where it went wrong 
            loss.backward()
            #adjust and make changes to optimizer 
            opt.step()
            #calculation for how many images the model got right 
            tr_l += loss.item()
            pred = torch.argmax(op.data, dim = 1)
            t_t += t.size(0)
            c_t += (pred == t).sum().item()
            
        model.eval()
        t_l = 0
        c_te = 0 
        tot_t = 0 
        
        #this no grad just tells to predict and not to update any gradients 
        with torch.no_grad(): 
            for d, t in test_l: 
                d, t = d.to(device), t.to(device)
                op = model(d)
                loss = crit(op, t)
                
                t_l += loss.item()
                tot_t += t.size(0)
                pred = torch.argmax(op.data , dim=1)
                c_te += (pred==t).sum().item()

        avg_tr_l = tr_l / len(train_l)
        tr_acc = (c_t / t_t) * 100 
        avg_te_l = t_l / len(test_l)
        tes_acc = (c_te/tot_t) * 100 
        print(f"Epoch: {e+1}, Train loss: {avg_tr_l:.4f}, Train Acc: {tr_acc:.2f}, Test Loss: {avg_te_l:.4f}, Test Acc: {tes_acc:.2f}")
            
device = torch.device("cpu")
model = CNNDropout()
crit = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr =0.001)
train(model, opt, crit, 30,device)

Epoch: 1, Train loss: 3.5126, Train Acc: 14.50, Test Loss: 2.3382, Test Acc: 12.00
Epoch: 2, Train loss: 2.2305, Train Acc: 26.00, Test Loss: 2.0000, Test Acc: 26.00
Epoch: 3, Train loss: 1.9518, Train Acc: 34.50, Test Loss: 2.1555, Test Acc: 26.00
Epoch: 4, Train loss: 1.6961, Train Acc: 38.00, Test Loss: 1.8727, Test Acc: 30.00
Epoch: 5, Train loss: 1.4993, Train Acc: 48.00, Test Loss: 2.1766, Test Acc: 26.00
Epoch: 6, Train loss: 1.2787, Train Acc: 54.00, Test Loss: 2.0553, Test Acc: 34.00
Epoch: 7, Train loss: 1.2054, Train Acc: 61.50, Test Loss: 2.1956, Test Acc: 28.00
Epoch: 8, Train loss: 1.0862, Train Acc: 64.00, Test Loss: 2.2926, Test Acc: 24.00
Epoch: 9, Train loss: 0.8775, Train Acc: 69.50, Test Loss: 2.2297, Test Acc: 26.00
Epoch: 10, Train loss: 0.8342, Train Acc: 73.00, Test Loss: 2.4472, Test Acc: 26.00
Epoch: 11, Train loss: 0.7619, Train Acc: 69.50, Test Loss: 2.0797, Test Acc: 34.00
Epoch: 12, Train loss: 0.6174, Train Acc: 81.00, Test Loss: 2.1539, Test Acc: 38.00
E